# Installing Packages

In [3]:
!pip cache purge
!pip install -q git+https://github.com/huggingface/diarizers.git
!pip install -q pyannote.audio accelerate
!pip install torch==2.9.1 torchaudio==2.9.1 torchvision==0.24.1 torchcodec==0.9.1 --index-url https://download.pytorch.org/whl/cu128
!pip install "numpy<2.0.0"
!pip install "datasets==4.0.0"

Files removed: 54
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 100.0 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires numba<0.63,>=0.60, but you have numba 0.63.1 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.0 which is incompatible.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is i

In [ ]:
import os
#WILL FORCE RESTART KERNEL
os._exit(0)

# IMPORTS

In [2]:
import pandas as pd
from datasets import Dataset, DatasetDict, Audio
from pathlib import Path
from diarizers import Preprocess
from diarizers import DataCollator
from diarizers import Metrics
from transformers import TrainingArguments
import torch
from pyannote.audio import Model
from diarizers import SegmentationModel
from transformers import Trainer

2026-02-14 06:15:30.831156: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771049731.001151     405 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771049731.051749     405 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771049731.465932     405 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771049731.465964     405 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771049731.465967     405 computation_placer.cc:177] computation placer alr

# FILE PATHS

In [3]:
train_base = Path("/kaggle/input/competitions/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/train")
audio_dir = train_base / "audio"
annotation_dir = train_base / "annotation"

# Loading The Training Dataset and Converting it to Callhome Type

In [4]:
def time_to_seconds(time_str):
    try:
        return pd.to_timedelta(time_str).total_seconds()
    except Exception:
        return float(time_str)

train_base = Path("/kaggle/input/competitions/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/train")
audio_dir = train_base / "audio"
annotation_dir = train_base / "annotation"

data_list = []
found_audio = list(audio_dir.glob("*.wav"))

for audio_path in audio_dir.glob("*.wav"):
    audio_id = audio_path.stem
    csv_path = annotation_dir / f"{audio_id}.csv"

    if csv_path.exists():
        # Load CSV
        df_ann = pd.read_csv(csv_path)
        df_ann.columns = df_ann.columns.str.strip().str.replace(' ', '_').str.lower()
        df_ann = df_ann.dropna(subset=['start_time', 'end_time', 'speaker_id'])

        starts = [time_to_seconds(t) for t in df_ann['start_time']]
        ends = [time_to_seconds(t) for t in df_ann['end_time']]
        speakers = df_ann['speaker_id'].astype(str).tolist()

        
        
        data_list.append({
            "audio": str(audio_path),
            "timestamps_start": starts,
            "timestamps_end": ends,
            "speakers": speakers
        })


train_ds = Dataset.from_list(data_list).cast_column("audio", Audio(sampling_rate=16000))
train_ds = train_ds.with_format("python") 
print(f"✅ Success! Loaded {len(train_ds)} files.")



✅ Success! Loaded 10 files.


In [5]:
split_dataset = train_ds.train_test_split(test_size=0.2, seed=0)
dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"]
})
dataset

DatasetDict({
    train: Dataset({
        features: ['audio', 'timestamps_start', 'timestamps_end', 'speakers'],
        num_rows: 8
    })
    validation: Dataset({
        features: ['audio', 'timestamps_start', 'timestamps_end', 'speakers'],
        num_rows: 2
    })
})

# LOADING SEGMENTATION 3.0 MODEL

In [6]:
model_path = "/kaggle/input/models/gomiie/pyannote-segmentation-3-0/pytorch/default/1/model_files/pytorch_model.bin"
pretrained = Model.from_pretrained(model_path)
model = SegmentationModel.from_pyannote_model(pretrained)
model.config

SegmentationModelConfig {
  "chunk_duration": 10.0,
  "max_speakers_per_chunk": 3,
  "max_speakers_per_frame": 2,
  "min_duration": null,
  "model_type": "pyannet",
  "sample_rate": 16000,
  "transformers_version": "4.57.1",
  "warm_up": [
    0.0,
    0.0
  ],
  "weigh_by_cardinality": false
}

# PREPROCESSING

In [11]:
import torchaudio

def official_logic_adapted(batch):
    # 1. Extract the FIRST item from the batch lists (index 0)
    # batch["audio"] is a list, e.g., [{"path": "file.wav"}]
    audio_entry = batch["audio"][0] 
    audio_path = audio_entry["path"]
    
    # 2. Load the 1-hour waveform (Uses ~230MB RAM instead of 27GB)
    waveform, sr = torchaudio.load(audio_path)
    
    # Ensure Mono for Whisper
    if waveform.shape[0] > 1: 
        waveform = waveform.mean(0, keepdim=True)
        
    # Resample to 16kHz
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        waveform = resampler(waveform)

    # 3. Format for the 'diarizers' preprocessor
    # We must wrap everything in a list [ ... ] so the library can do file["audio"][0]
    file_payload = {
        "audio": [{
            "array": waveform.numpy().flatten(), 
            "sampling_rate": 16000
        }],
        "timestamps_start": [batch["timestamps_start"][0]], # Access 0-th batch item
        "timestamps_end": [batch["timestamps_end"][0]],
        "speakers": [batch["speakers"][0]]
    }

    # 4. Process (Slices the 1-hour file into ~120 chunks)
    # overlap=0.5 matches the official example
    return preprocessor(file_payload, random=False, overlap=0.8)

def official_logic_adapted2(batch):
    # 1. Extract the FIRST item from the batch lists (index 0)
    # batch["audio"] is a list, e.g., [{"path": "file.wav"}]
    audio_entry = batch["audio"][0] 
    audio_path = audio_entry["path"]
    
    # 2. Load the 1-hour waveform (Uses ~230MB RAM instead of 27GB)
    waveform, sr = torchaudio.load(audio_path)
    
    # Ensure Mono for Whisper
    if waveform.shape[0] > 1: 
        waveform = waveform.mean(0, keepdim=True)
        
    # Resample to 16kHz
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        waveform = resampler(waveform)

    # 3. Format for the 'diarizers' preprocessor
    # We must wrap everything in a list [ ... ] so the library can do file["audio"][0]
    file_payload = {
        "audio": [{
            "array": waveform.numpy().flatten(), 
            "sampling_rate": 16000
        }],
        "timestamps_start": [batch["timestamps_start"][0]], # Access 0-th batch item
        "timestamps_end": [batch["timestamps_end"][0]],
        "speakers": [batch["speakers"][0]]
    }

    # 4. Process (Slices the 1-hour file into ~120 chunks)
    # overlap=0.5 matches the official example
    return preprocessor(file_payload, random=False, overlap=0.0)

In [12]:
preprocessor = Preprocess(model.config)
for split in dataset.keys():
    dataset[split] = dataset[split].cast_column("audio", Audio(decode=False))
train_set = dataset['train'].map(
    official_logic_adapted,
    batched=True,
    batch_size = 2,
    remove_columns=dataset['train'].column_names,
    load_from_cache_file=False
).with_format("torch")

val_set = dataset['validation'].map(
    official_logic_adapted2,
    batched=True,
    batch_size = 2,
    remove_columns=dataset['validation'].column_names,
    load_from_cache_file=False
).with_format('torch')

train_set = train_set.shuffle(seed=42)

/usr/local/lib/python3.12/dist-packages/asteroid_filterbanks/enc_dec.py:202: UserWarning: Input tensor was 2D. Applying the corresponding Decoder to the current output will result in a 3D tensor. This behaviours was introduced to match Conv1D and ConvTranspose1D, please use 3D inputs to avoid it. For example, this can be done with input_tensor.unsqueeze(1).
  warnings.warn(


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

# SANITY CHECK

In [14]:
# Select the first processed example
print(len(train_set))
test_idx = 100 
sample = train_set[test_idx]


print("--- Kaggle Processed Chunk Check ---")
print(f"Keys available in sample: {sample.keys()}") # Just to be 100% sure
print(f"Waveform Shape: {sample['waveforms'].shape}")
print(f"Waveform Max:   {sample['waveforms'].max():.4f}")
print(f"Waveform Mean:  {sample['waveforms'].mean():.4f}")
print("")
print(f"Label Matrix Shape: {sample['labels'].shape}")
print(f"Label Sum (Speech):  {sample['labels'].sum()}")


7361
--- Kaggle Processed Chunk Check ---
Keys available in sample: dict_keys(['waveforms', 'labels', 'nb_speakers'])
Waveform Shape: torch.Size([160000])
Waveform Max:   0.3452
Waveform Mean:  -0.0000

Label Matrix Shape: torch.Size([589, 2])
Label Sum (Speech):  590


# TRAINING DEFINITION

In [16]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Get the token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login
login(token=hf_token)

data_collator = DataCollator(max_speakers_per_chunk=model.config.max_speakers_per_chunk)
metrics = Metrics(model.specifications)
metrics.metrics

{'der': DiarizationErrorRate(),
 'confusion': SpeakerConfusionRate(),
 'missed_detection': MissedDetectionRate(),
 'false_alarm': FalseAlarmRate()}

In [23]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
training_args = TrainingArguments(
    output_dir= '/kaggle/working/BengaliDiarization',
    #hub_model_id="gomiiie/BengaliDiarization-v1", # <--- Add this!
    save_strategy="steps",
    warmup_steps=100,
    learning_rate=8e-5,
    num_train_epochs=20,
    lr_scheduler_type="linear",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    eval_strategy="steps",
    gradient_accumulation_steps=1,
    dataloader_num_workers=0,
    logging_steps=5,
    eval_steps=100,
    save_steps=100,
    load_best_model_at_end=True,
    seed=42,
    push_to_hub=False,
    fp16=True,
    no_cuda=False,
    greater_is_better=False,
    save_total_limit=2,  
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_set,
    data_collator=data_collator,
    eval_dataset=val_set,
    compute_metrics=metrics,
)

# TRAIN

In [24]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss,Validation Loss,Der,False Alarm,Missed Detection,Confusion
100,0.556500,0.322021,0.115225,0.015784,0.062809,0.036633
200,0.498000,0.300946,0.105512,0.016364,0.047351,0.041797
300,0.495500,0.286337,0.097938,0.016102,0.043098,0.038738
400,0.476600,0.278443,0.098140,0.016158,0.042905,0.039077
500,0.472600,0.275516,0.097027,0.015084,0.043459,0.038484
600,0.457400,0.273990,0.096572,0.014461,0.044967,0.037144
700,0.444100,0.271891,0.095648,0.014250,0.044512,0.036886
800,0.415700,0.273580,0.095811,0.014130,0.044881,0.036800
900,0.491600,0.272187,0.095429,0.014057,0.045264,0.036109
1000,0.435800,0.268202,0.094140,0.014065,0.044834,0.035241


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

TrainOutput(global_step=1160, training_loss=0.4824239993917531, metrics={'train_runtime': 1741.0255, 'train_samples_per_second': 84.559, 'train_steps_per_second': 0.666, 'total_flos': 0.0, 'train_loss': 0.4824239993917531, 'epoch': 20.0})

# SAVE

In [26]:
trainer.save_model("/kaggle/working/fine_tuned_segmentation")

model.config.save_pretrained("/kaggle/working/fine_tuned_segmentation")

try:
    preprocessor.feature_extractor.save_pretrained("/kaggle/working/fine_tuned_segmentation")
except:
    print("No separate feature extractor to save, ignoring.")

print("✅ Model and Config saved successfully!")

No separate feature extractor to save, ignoring.
✅ Model and Config saved successfully!


In [31]:
repo_id = "gomiiie/BengaliDiarization-v2"

# This will create the repo if it doesn't exist
trainer.push_to_hub(
    commit_message="Final 20 epoch version",
    token=hf_token
)

print("DONE")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

DONE
